# 实验三：图像关键点检测及图像拼接

参考 [CV_exp / 3rd.exp](https://github.com/Guyudong1/CV_exp/tree/main/3rd.exp)，使用用户提供的插画完成特征提取、匹配、单应矩阵估计和图像拼接。算法实现见 `experiment.py`，本 Notebook 调用同一份代码，并展示各阶段结果。

输入为 `assets/input.jpg`。先缩小到工作分辨率，再裁出两幅重叠视图；随后分别改变图像 B 的尺寸、亮度和透视关系。**这些图像来自同一张图片，是受控模拟实验。** 实验结果用于验证配准流程，不能据此判断真实多视角照片中的视差、遮挡和相机运动问题。

在仓库根目录执行 `.venv\Scripts\python.exe -m pip install -r exp3/requirements.txt`，再选择 `.venv` 的 Python 内核。可从仓库根目录或 `exp3` 目录启动，按顺序运行全部单元格。运行会重新生成 `img/` 中的实验视图和 `result/` 中的正式结果。

In [ ]:
from pathlib import Path
import csv
import inspect
import sys

import cv2
import numpy as np
from IPython.display import Image, Markdown, display

root = Path.cwd()
exp_dir = root / "exp3" if (root / "exp3" / "experiment.py").is_file() else root
assert (exp_dir / "experiment.py").is_file(), "请从仓库根目录或 exp3 目录启动。"
if str(exp_dir) not in sys.path:
    sys.path.insert(0, str(exp_dir))
from experiment import read_image, run_experiment, detect_and_match, stitch_pair

input_path = exp_dir / "assets" / "input.jpg"
output_dir = exp_dir / "result"
LONG_EDGE = 1800
SEED = 2026


## 1. 读取图片并生成重叠视图

`read_image` 读取文件字节，再用 OpenCV 解码，支持中文路径。OpenCV 的彩色数组按 BGR 排列，特征检测使用灰度图。灰度图保留结构和亮度信息，减少特征提取的输入通道数。

下面运行整套实验，后续单元格读取同一次运行的结果。输入图片保持原文件；工作副本用于控制计算量。裁剪位置与几何变换由程序固定，因此可以计算预期单应矩阵，检查估计误差。

In [ ]:
original = read_image(input_path)
summary = run_experiment(input_path, output_dir, long_edge=LONG_EDGE, seed=SEED)
info = summary["input"]
print(f"原图：宽 {info['width']} × 高 {info['height']}，数组形状 {original.shape}")
print(f"工作图：宽 {info['working_width']} × 高 {info['working_height']}")
print("输入 SHA-256：", info["sha256"])
print("随机种子：", summary["parameters"]["seed"])
display(Image(filename=str(output_dir / "input_views.png"), width=1100))


## 2. SIFT 特征提取

尺度不变特征变换（Scale-Invariant Feature Transform，SIFT）在灰度图上检测关键点，并为每个点计算描述符。关键点包含位置、尺度和方向；描述符用于比较不同图像中的局部结构。

关键点数量反映当前参数下检测到的局部特征数量。能否用于配准，还要看描述符匹配和几何验证。下图显示基准视图中的关键点。

In [ ]:
cases = {case["id"]: case for case in summary["cases"]}
baseline = cases["baseline"]
print("图像 A 关键点：", baseline["keypoints_a"])
print("图像 B 关键点：", baseline["keypoints_b"])
display(Image(filename=str(output_dir / "sift_keypoints.png"), width=1100))


## 3. FLANN 匹配与 Lowe 比率筛选

近似最近邻快速搜索库（Fast Library for Approximate Nearest Neighbors，FLANN）为图像 A 的每个描述符寻找图像 B 中的两个近邻。沿用参考实验的参数：KD 树数量为 5，搜索检查次数为 50，`k=2`。

若最近邻距离 `m.distance` 小于次近邻距离 `n.distance` 的 0.7 倍，则保留该匹配。比率筛选排除描述符距离难以区分的候选点。保留下来的匹配仍可能含有错误，需要 RANSAC 检验几何关系。

下方展示实际调用的源码，避免在 Notebook 中维护另一份算法实现。

In [ ]:
display(Markdown("```python\n" + inspect.getsource(detect_and_match) + "\n```"))
params = summary["parameters"]
print("FLANN trees / checks：", params["flann_trees"], "/", params["flann_checks"])
print("Lowe 比率阈值：", params["ratio_threshold"])
print("初始近邻匹配组数：", baseline["knn_pairs"])
print("比率筛选后匹配数：", baseline["good_matches"])
display(Image(filename=str(output_dir / "matches_baseline.jpg"), width=1100))


## 4. RANSAC 与单应矩阵

随机采样一致性算法（Random Sample Consensus，RANSAC）从匹配点中估计单应矩阵，并识别满足该几何模型的内点。重投影阈值为 5 个工作图像像素。至少需要 4 对满足非退化几何条件的匹配点，才能估计单应矩阵。

这里把图像 B 的坐标映射到图像 A：`p_A ∼ H @ p_B`。齐次坐标计算后，需要除以第三个分量。内点重投影误差衡量估计模型与所选匹配的一致性；角点误差则与构造视图时的已知变换比较。两种误差的参考对象不同。

In [ ]:
estimated = np.asarray(baseline["homography_b_to_a"])
expected = np.asarray(baseline["expected_homography_b_to_a"])
np.set_printoptions(precision=6, suppress=True)
print("估计的 B → A 单应矩阵：\n", estimated)
print("构造视图得到的预期矩阵：\n", expected)
print(f"RANSAC 内点：{baseline['inliers']} / {baseline['good_matches']}")
print(f"内点比例：{baseline['inlier_ratio']:.2%}")
print(f"内点重投影 RMSE：{baseline['inlier_rmse_px']:.4f} px")
print(f"角点映射 RMSE：{baseline['corner_rmse_px']:.4f} px")


## 5. 画布平移、有效掩膜与羽化融合

先把图像 B 的角点变换到图像 A 的坐标系，再合并两图的边界，确定输出画布。若变换后出现负坐标，就加入平移矩阵 `T`，用 `T @ H` 把图像 B 映射到画布中。

有效掩膜标记图像实际覆盖的像素。图像内容中的黑色像素也可以有效，因此不能用“像素是否为黑色”判断覆盖区域。重叠区域根据到有效边界的距离计算羽化权重；单图覆盖区域保留对应图像。

亮度补偿利用对齐后的重叠区域估计光度关系。它可以减轻这次线性亮度变化造成的接缝，但不能恢复裁剪和饱和造成的信息损失。

In [ ]:
display(Markdown("```python\n" + inspect.getsource(stitch_pair) + "\n```"))
print(f"基准拼接尺寸：{baseline['panorama_width']} × {baseline['panorama_height']}")
print(f"重叠区域补偿前 MAE：{baseline['overlap_mae_raw']:.4f}")
print(f"重叠区域补偿后 MAE：{baseline['overlap_mae_corrected']:.4f}")
display(Image(filename=str(output_dir / "panorama_baseline.png"), width=650))


## 6. 尺寸、亮度与透视变化

固定图像 A，分别改变图像 B，每组重新提取特征、匹配和估计单应矩阵。尺寸变化检查尺度适应性；亮度变化检查描述符与光度补偿；透视变化检查单应矩阵对投影变换的估计。

下面的结果来自本次运行。图像 B 的变换方式和参数保存在 `summary.json`。匹配图与拼接图一起观察，能区分几何错位和亮度接缝。

In [ ]:
for case_id in ("scale", "brightness", "perspective"):
    case = cases[case_id]
    display(Markdown(f"### {case['label']}"))
    print(f"优质匹配 {case['good_matches']}，内点 {case['inliers']}，内点比例 {case['inlier_ratio']:.2%}")
    print(f"内点 RMSE {case['inlier_rmse_px']:.4f} px，角点 RMSE {case['corner_rmse_px']:.4f} px")
    display(Image(filename=str(output_dir / f"matches_{case_id}.jpg"), width=1000))
    display(Image(filename=str(output_dir / f"panorama_{case_id}.png"), width=600))

display(Markdown("### 亮度组的融合对比"))
display(Image(filename=str(output_dir / "brightness_comparison.png"), width=1100))


## 7. 指标与结果分析

表中内点比例为 RANSAC 内点数除以比率筛选后的匹配数。内点 RMSE 和角点 RMSE 的单位均为工作图像像素。重叠区域平均绝对误差（Mean Absolute Error，MAE）在对齐后的彩色通道值上计算，数值越小表示重叠区域的像素差异越小。

亮度补偿直接利用重叠区域拟合光度参数，所以该区域内 MAE 的下降属于拟合后的诊断结果。它不能作为独立泛化性能证据。几何准确性还需结合已知变换的角点误差判断。

In [ ]:
table = [
    "| 组别 | 优质匹配 | 内点 | 内点比例 | 内点 RMSE / px | 角点 RMSE / px | 原始 MAE | 补偿后 MAE |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for case in summary["cases"]:
    table.append(
        f"| {case['label']} | {case['good_matches']} | {case['inliers']} | "
        f"{case['inlier_ratio']:.2%} | {case['inlier_rmse_px']:.4f} | "
        f"{case['corner_rmse_px']:.4f} | {case['overlap_mae_raw']:.4f} | "
        f"{case['overlap_mae_corrected']:.4f} |"
    )
display(Markdown("\n".join(table)))
with (output_dir / "metrics.csv").open(encoding="utf-8", newline="") as handle:
    csv_rows = list(csv.DictReader(handle))
print("CSV 指标记录数：", len(csv_rows))
display(Image(filename=str(output_dir / "overview.png"), width=1100))


## 8. 验证结论与适用范围

下面列出主程序的实际验证结果。所有检查通过，表示本次运行满足程序设定的几何、输出和数值条件；具体含义以检查名称和报告为准。

这组图片共享同一个原图，重叠部分不存在真实视差，也没有独立拍摄造成的运动和遮挡。单应矩阵适合描述平面或纯旋转等条件下的投影关系。真实场景中的景深变化、非刚体运动、镜头畸变和局部曝光差异，需要另行采样与验证。

In [ ]:
for name, passed in summary["validation"].items():
    print(f"{name}: {passed}")
assert all(summary["validation"].values()), "存在未通过的检查，请查看 summary.json。"
print("正式结果目录：", output_dir.resolve())
print("实验参数和矩阵：", (output_dir / "summary.json").resolve())
print("逐组指标：", (output_dir / "metrics.csv").resolve())
